https://ccrma.stanford.edu/~jos/pasp/

http://mikedolbear.com/seriously-wired/modelled-drum-sounds/

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter
from scipy.signal import convolve
from scipy.io.wavfile import write
from IPython.display import Audio

In [ ]:
musical_alphabet = ["C", "C#/Db", "D", "Eb/D#", "E", "F", "F#/Gb", "G", "Ab/G#", "A", "Bb/A#", "B"]
frequencies = [440*(2**((m - 69)/12)) for m in range(12, 120)]
notes_lut = {}
octave = -1
for index, frequency in enumerate(frequencies):
    index %= 12
    if index == 0:
        octave +=1
    note = musical_alphabet[index] + str(octave)
    notes_lut[note] = frequency
notes_full_arr = np.array(list(notes_lut.keys()))

def semitone_shift(note, semitones):
    if note is not None:
        return notes_full_arr[np.where(notes_full_arr == note)[0] + semitones][0]
    else:
        return None

def harmonics_calculator(base_frequncy, no_harmonics):
    harmonics = []
    for harmonic in range(1, no_harmonics + 1):
        harmonic_frequency = base_frequncy*harmonic
        frequency_difference = np.inf
        for note, note_frequency in notes_lut.items():
            if abs(harmonic_frequency - note_frequency) < frequency_difference:
                frequency_difference = abs(harmonic_frequency - note_frequency)
                harmonic_note = note
        harmonics.append((harmonic_note, harmonic_frequency, notes_lut[harmonic_note]))
    return harmonics

def generate_pitch(note, duration, sampling_rate = 44100):
    if note is not None:
        return np.sin(2*np.pi*notes_lut[note]*np.linspace(0, duration, int(sampling_rate*duration)))
    else:
        return np.zeros((int(sampling_rate*duration)))
    
def wobbly_piano(note_0, note_1, duration, filter_width, sample_rate):
    if note_0 is not None and note_1 is not None:
        p_0 = generate_pitch(note_0, duration, sample_rate) + generate_pitch(harmonics_calculator(notes_lut[note_0], 5)[-1][0], duration, sample_rate) + generate_pitch(harmonics_calculator(notes_lut[note_0], 4)[-1][0], duration, sample_rate)
        p_1 = generate_pitch(note_1, duration, sample_rate) + generate_pitch(harmonics_calculator(notes_lut[note_1], 3)[-1][0], duration, sample_rate) + generate_pitch(harmonics_calculator(notes_lut[note_1], 2)[-1][0], duration, sample_rate)
        e = np.exp(np.linspace(1, -1, p_0.size))
        return convolve((p_0*p_1*e)**3, ((1 - np.cos(np.linspace(0, 2*np.pi, filter_width)))/2)**3, mode = "same")
    else:
        return np.zeros(int(duration*sample_rate))

def adsr(a = 1, d = 1, s = 1, r = 1, s_level = 0.5, duration = 1, amplitude = 1, sampling_rate = 44100):
    if duration != 0:
        total = np.sum([a, d, s, r])
        a /= total
        d /= total
        s /= total
        r /= total
        attack = gaussian_filter(np.linspace(0, 1, int(sampling_rate*duration*a)), 1000)
        attack -= attack.min()
        attack /= attack.max()
        decay = gaussian_filter(np.linspace(1, 0, int(sampling_rate*duration*d)), 1000)
        decay -= decay.min()
        decay /= decay.max()/(1 - s_level)
        decay += s_level
        sustain = np.full(int(sampling_rate*duration*s), 1, dtype = float)
        sustain *= s_level
        release = gaussian_filter(np.linspace(1, 0, int(sampling_rate*duration*r)), 1000)
        release -= release.min()
        release /= release.max()/s_level
        envelope = np.concatenate([attack, decay, sustain, release])
        if envelope.shape[0] != int(duration*sampling_rate):
            envelope = np.concatenate([envelope, np.zeros((np.abs(int(duration*sampling_rate) - envelope.shape[0])))])
        return envelope*amplitude
    else:
        return np.zeros(1)

def convert_bars(bars_input, note_values_input, bpm = 90, time_signature = (4, 4), semitones = 0):
    for note_values_index, note_values in enumerate(note_values_input):
        if sum(note_values) < time_signature[0]/time_signature[1]:
            note_values.append(time_signature[0]/time_signature[1] - sum(note_values))
            bars_input[note_values_index].append(None)
        elif sum(note_values) > time_signature[0]/time_signature[1]:
            raise Exception("Too many notes in the bar!")
    bars = np.full((len(bars_input), max([len(l) for l in bars_input])), None)
    for bar_index, bar in enumerate(bars_input):
        bars[bar_index, :len(bar)] = bar
    note_values = np.full_like(bars, 0, dtype = float)
    for note_value_index, note_value in enumerate(note_values_input):
        note_values[note_value_index, :len(note_value)] = note_value
    note_durations = note_values*time_signature[1]*60/bpm
    if semitones != 0:
        for index in np.ndindex(bars.shape):
            bars[index] = semitone_shift(bars[index], semitones)
    return bars, note_durations

def generate_wave(bars, note_durations, envelope_params, sampling_rate):
    total_samples = int(sampling_rate*note_durations.sum())
    wave = np.concatenate([generate_pitch(note, note_durations[index], sampling_rate)*adsr(*envelope_params, note_durations[index], 1, sampling_rate) for index, note in np.ndenumerate(bars)])
    if wave.size < total_samples:
        wave = np.concatenate([wave, np.zeros((np.abs(wave.size - total_samples)))])
    elif wave.size > total_samples:
        wave = wave[:-np.abs(wave.size - total_samples)]
    return wave

In [ ]:
bpm = 90
whole_note_duration = 60/bpm
sampling_rate = 44100
channels = np.array([["E3", "E3", "G3", "F3", "F3", None],
                     ["C4", "C5", None, "D5", "D5", "C3"],
                     ["E4", "E4", "B4", "B4", "B5", "E4"],
                     [None, "A4", "G4", "C4", "C4", "A4"]])
note_durations = whole_note_duration/np.array([1, 1, 2, 4, 4, 2])
total_duration = note_durations.sum()
total_samples = int(sampling_rate*total_duration)
envelopes = [adsr(1, 2, 1, 0.5, 0.8, note_duration, 44100) for note_duration in note_durations]
waves = np.zeros((channels.shape[0], total_samples))
for channel_index, channel in enumerate(channels):
    pitches = []
    for note_index, note in enumerate(channel):
        pitch = generate_pitch(note, note_durations[note_index], sampling_rate)*envelopes[note_index]
        pitches.append(pitch)
    wave = np.concatenate(pitches)
    if wave.size < total_samples:
        wave = np.concatenate([wave, np.zeros((np.abs(wave.size - total_samples)))])
    elif wave.size > total_samples:
        wave = wave[:-np.abs(wave.size - total_samples)]
    waves[channel_index] = wave

Audio(waves, rate = sampling_rate)

In [ ]:
bpm = 60
whole_note_duration = 60/bpm
sampling_rate = 44100
channels = np.array([["A4", "B4", "C5", "D5", "E5", "F5", "G5", "A5"],
                     [None, None, None, None, None, None, None, None],
                     [None, None, None, None, None, None, None, None],
                     [None, None, None, None, None, None, None, None]])
note_durations = whole_note_duration/np.array([1, 1, 1, 1, 1, 1, 1, 1])
total_duration = note_durations.sum()
total_samples = int(sampling_rate*total_duration)
envelopes = [adsr(1, 2, 1, 0.5, 0.8, note_duration, 44100) for note_duration in note_durations]
waves = np.zeros((channels.shape[0], total_samples))
for channel_index, channel in enumerate(channels):
    pitches = []
    for note_index, note in enumerate(channel):
        pitch = generate_pitch(note, note_durations[note_index], sampling_rate)*envelopes[note_index]
        pitches.append(pitch)
    wave = np.concatenate(pitches)
    if wave.size < total_samples:
        wave = np.concatenate([wave, np.zeros((np.abs(wave.size - total_samples)))])
    elif wave.size > total_samples:
        wave = wave[:-np.abs(wave.size - total_samples)]
    waves[channel_index] = wave

Audio(waves, rate = sampling_rate)

In [ ]:
bpm = 90
time_signature = (4, 4)
sampling_rate = 48000
repeats = 4
channels = np.array([["A4", "F4", "D4", "D4", "G4", "G4"],
                     ["C5", "A4", "F4", None, "B4", "B4"],
                     ["E5", "C5", "A4", None, "D5", "D5"],
                     [None, "E5", "D5", None, "F5", None]])
note_values = np.array([1/16, 1/2, 1/4, 1/16, 1/16, 1/16])
if note_values.sum() < time_signature[0]/time_signature[1]:
    note_values = np.concatenate([note_values, np.array([time_signature[0]/time_signature[1] - note_values.sum()])])
    channels = np.concatenate([channels, np.full((channels.shape[0], 1), None)], axis = 1)
elif note_values.sum() > time_signature[0]/time_signature[1]:
    raise Exception("Too many notes in the bar!")
note_durations = note_values*time_signature[1]*60/bpm
note_durations += np.random.uniform(-0.025, 0.025, note_durations.size)
total_duration = note_durations.sum()
total_samples = int(sampling_rate*total_duration)
envelopes = [adsr(0.25, 1, 0.75, 0.5, 0.8, note_durations[0], 1, sampling_rate),
             adsr(0.5, 0.5, 0.25, 0.75, 0.6, note_durations[1], 0.75, sampling_rate),
             adsr(0.25, 1, 0.75, 0.5, 0.7, note_durations[2], 0.65, sampling_rate),
             adsr(duration = note_durations[3], sampling_rate = sampling_rate),
             adsr(0.5, 0.5, 0.25, 0.75, 0.6, note_durations[4], 0.65, sampling_rate),
             adsr(0.25, 0.75, 0.25, 0.5, 0.6, note_durations[5], 0.75, sampling_rate)]
if len(envelopes) != note_values.size:
    envelopes.append(1)
waves = np.zeros((channels.shape[0], total_samples))
for channel_index, channel in enumerate(channels):
    pitches = []
    for note_index, note_0 in enumerate(channel):
        pitch = generate_pitch(note, note_durations[note_index], sampling_rate)*envelopes[note_index]
        note_1 = None if note_0 is None else harmonics_calculator(notes_lut[note_0], 3)[-1][0]
        pitch_1 = wobbly_piano(note_0, note_1, note_durations[note_index], 256, sampling_rate)
        note_2 = None if note_0 is None else harmonics_calculator(notes_lut[note_0], 4)[-1][0]
        pitch_2 = wobbly_piano(note_0, note_2, note_durations[note_index], 64, sampling_rate)
        pitch += np.sqrt(1.5*pitch_1**2 + 0.5*pitch_2**2)
        pitch += convolve(pitch, ((1 - np.cos(np.linspace(0, 2*np.pi, 64)))/2), mode = "same")
        if note_0 is not None:
            pitch += convolve(pitch, np.sin(np.linspace(0, notes_lut[note_0]*np.pi, 512)), mode = "same")
        pitch *= envelopes[note_index]
        pitches.append(pitch)
    wave = np.concatenate(pitches)
    if wave.size < total_samples:
        wave = np.concatenate([wave, np.zeros((np.abs(wave.size - total_samples)))])
    elif wave.size > total_samples:
        wave = wave[:-np.abs(wave.size - total_samples)]
    waves[channel_index] = wave
waves = np.concatenate([waves for _ in range(repeats)], axis = 1)

Audio(waves, rate = sampling_rate)

In [ ]:
bpm = 106
time_signature = (4, 4)
sampling_rate = 48000
repeats = 3
bars_input = [["C4", "G4", "G4", "Eb/D#4"], ["D4", "D4", "D4", "D4", "Eb/D#4"]]
note_values_input = [[1/4, 1/4, 1/4, 1/4], [1/4, 1/4, 1/4, 1/8, 1/8]]
envelope_params = (1/8, 1/2, 1/8, 1/4, 0.8)
for note_values_index, note_values in enumerate(note_values_input):
    if sum(note_values) < time_signature[0]/time_signature[1]:
        note_values.append(time_signature[0]/time_signature[1] - sum(note_values))
        bars_input[note_values_index].append(None)
    elif sum(note_values) > time_signature[0]/time_signature[1]:
        raise Exception("Too many notes in the bar!")
bars = np.full((len(bars_input), max([len(l) for l in bars_input])), None)
for bar_index, bar in enumerate(bars_input):
    bars[bar_index, :len(bar)] = bar
note_values = np.full_like(bars, 0, dtype = float)
for note_value_index, note_value in enumerate(note_values_input):
    note_values[note_value_index, :len(note_value)] = note_value
note_durations = note_values*time_signature[1]*60/bpm
total_samples = int(sampling_rate*note_durations.sum())


wave = np.concatenate([generate_pitch(note, note_durations[index], sampling_rate)*adsr(*envelope_params, note_durations[index], 1, sampling_rate) for index, note in np.ndenumerate(bars)])
if wave.size < total_samples:
    wave = np.concatenate([wave, np.zeros((np.abs(wave.size - total_samples)))])
elif wave.size > total_samples:
    wave = wave[:-np.abs(wave.size - total_samples)]
wave = np.concatenate([wave for _ in range(repeats)])

Audio(wave, rate = sampling_rate)

In [ ]:
bpm = 103
time_signature = (4, 4)
sampling_rate = 48000
repeats = 3
envelope_params = (1/8, 1/2, 1/8, 1/4, 0.8)
bars = [[["C5", "G5", "G5", "Eb/D#5"], ["D5", "D5", "D5", "D5", "Eb/D#5"]],
        [["C4", "Bb/A#3"], ["Ab/G#3", "Bb/A#3"]],
        [["Eb/D#4", "D4"], ["C4", "D4"]],
        [["G4", "F4"], ["Eb/D#4", "F4"]]]
note_values = [[[1/4, 1/4, 1/4, 1/4], [1/4, 1/4, 1/4, 1/8, 1/8]],
               [[1/2, 1/2], [1/2, 1/2]],
               [[1/2, 1/2], [1/2, 1/2]],
               [[1/2, 1/2], [1/2, 1/2]]]
waves = [generate_wave(*convert_bars(bars[i], note_values[i], bpm, time_signature, 0), envelope_params, sampling_rate) for i in range(len(bars))]
waves = np.concatenate([waves for _ in range(repeats)], axis = 1)

Audio(waves, rate = sampling_rate)

In [ ]:
bpm = 120
time_signature = (4, 4)
sampling_rate = 48000
repeats = 2
envelope_params = (1/4, 1/2, 1/8, 1/8, 0.6)
bars = [[["Bb/A#4", "G4", "Eb/D#4", "C4"], ["Ab/G#4", "F4", "D4"], ["G4", "Eb/D#4", "C4", "Ab/G#3"], ["F4", "D4", "Bb/A#3"]],
        [["C3", "Bb/A#2"], ["Bb/A#2", "Ab/G#2"], ["Ab/G#2", "G3"], ["G2", "C3"]],
        [["Eb/D#3", "D3"], ["D3", "C3"], ["C3", "Bb/A#2"], ["Bb/A#2", "Eb/D#3"]],
        [["G3", "F3"], ["F3", "Eb/D#3"], ["Eb/D#3", "D3"], ["D3", "G3"]]]
note_values = [[[1/4, 1/8, 1/8, 1/2], [1/2, 1/4, 1/4], [1/4, 1/8, 1/8, 1/2], [1/4, 1/4, 1/2]],
               [[1/2, 1/2], [1/2, 1/2], [1/2, 1/2], [1/2, 1/2]],
               [[1/2, 1/2], [1/2, 1/2], [1/2, 1/2], [1/2, 1/2]],
               [[1/2, 1/2], [1/2, 1/2], [1/2, 1/2], [1/2, 1/2]]]
waves = [generate_wave(*convert_bars(bars[i], note_values[i], bpm, time_signature, 6), envelope_params, sampling_rate) for i in range(len(bars))]
waves = np.concatenate([waves for _ in range(repeats)], axis = 1)

Audio(waves, rate = sampling_rate)